# MCMC Convergence Diagnostics — Synthetic Data

Demonstrates convergence of the LG generation algorithm using controlled
synthetic experiments.  We first run a long "reference" MCMC chain to
establish the equilibrium, then launch **multiple independent chains**
from very different random initial densities and show they all converge
to the same equilibrium (spectral distance, edge count, degree-distribution
KS statistic, and the adjacency-ESD KL divergence all collapse).

The adjacency-ESD KL panel `D_KL(rho_t || rho_ref)` tracks the *same* object
Algorithm 1's stopping criterion monitors, so the figure directly illustrates
the criterion.

**Design choice:** We use `d=0` (S_i = deg(i)) so the feedback is linear
and sigma=-6 produces a genuinely sparse equilibrium.  With `d>=1` the
neighbourhood sums grow quadratically with degree, requiring much more
negative sigma to stay sparse.

In [6]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append('../..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import ks_2samp
from scipy.special import expit
from collections import deque

import src.logit_graph.graph as graph

plt.style.use('seaborn-v0_8-white')
mpl.rcParams.update({
    'font.family': 'serif',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from scipy.stats import entropy


def _adjacency_esd(adj, bin_edges):
    """Empirical spectral density (ESD) of the ADJACENCY matrix: density
    histogram of A's eigenvalues over a fixed grid (eigenvalues clipped so
    transiently-denser chains keep all their spectral mass)."""
    eig = np.linalg.eigvalsh(adj)
    eig = np.clip(eig, bin_edges[0], bin_edges[-1])
    hist, _ = np.histogram(eig, bins=bin_edges, density=True)
    return hist


def _esd_kl(cur_den, ref_den):
    """KL divergence D_KL(rho_t || rho_ref) between adjacency ESDs (matches
    gic.py calculate_spectral_distance with dist='KL')."""
    return float(entropy(cur_den + 1e-10, ref_den + 1e-10))


def run_chain(n, d, sigma, max_iter, check_interval, er_p,
              ref_spectrum, ref_degrees, ref_esd, esd_bins):
    """Run one MCMC chain and record diagnostics at every check_interval."""
    gm = graph.GraphModel(n=n, d=d, sigma=sigma, er_p=er_p)

    spec_dists = []
    edge_counts = []
    ks_stats = []
    esd_kls = []

    for i in range(max_iter):
        gm.add_remove_edge()

        if i % check_interval == 0:
            cur_spec = graph.GraphModel.calculate_spectrum(gm.graph)
            spec_dists.append(float(np.linalg.norm(cur_spec - ref_spectrum)))
            edge_counts.append(gm._edge_count)
            cur_deg = gm.graph.sum(axis=1)
            ks_stat, _ = ks_2samp(cur_deg, ref_degrees)
            ks_stats.append(ks_stat)
            esd_kls.append(_esd_kl(_adjacency_esd(gm.graph, esd_bins), ref_esd))

    return spec_dists, edge_counts, ks_stats, esd_kls

In [ ]:
# --- Experiment parameters ---
# d=0 keeps feedback linear (S_i = deg(i)), so sigma=-6 yields a sparse graph.
# With d>=1 the neighbourhood sums grow quadratically with degree,
# requiring much more negative sigma to avoid saturating to a complete graph.
N = 750
D = 0
SIGMA = -2.0
MAX_ITER = 1_000_000 # per chain (fast with d=0)
CHECK_INTERVAL = 500

# Generate a well-equilibrated reference graph by running a long chain.
# Use the same fixed-iteration approach as the test chains so both
# sample from the true stationary distribution.
print('Generating reference (long-run equilibrium) graph ...')
gt_model = graph.GraphModel(n=N, d=D, sigma=SIGMA, er_p=0.05)
for _ in range(MAX_ITER):
    gt_model.add_remove_edge()
gt_spectrum = graph.GraphModel.calculate_spectrum(gt_model.graph)
gt_degrees = gt_model.graph.sum(axis=1)
# Adjacency-ESD reference: fix a bin grid spanning the reference adjacency
# eigenvalues (with padding), then take its density as rho_ref. This is the
# object Algorithm 1's stopping criterion monitors (KL to the reference).
gt_eig = np.linalg.eigvalsh(gt_model.graph)
lo, hi = float(gt_eig.min()), float(gt_eig.max())
pad = 0.05 * (hi - lo) + 1e-9
esd_bins = np.linspace(lo - pad, hi + pad, 51)  # 50 bins
gt_esd = _adjacency_esd(gt_model.graph, esd_bins)
gt_edges = gt_model._edge_count
gt_density = gt_edges / (N * (N - 1) / 2)
print(f'Reference: {gt_edges} edges, mean degree {gt_degrees.mean():.2f}, '
      f'density {gt_density:.4f}')

In [ ]:
# Run N_CHAINS independent chains from different random ER seeds
results = []
er_ps = [0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.075]  # varied initial densities
N_CHAINS = len(er_ps)

for chain_id in range(N_CHAINS):
    p0 = er_ps[chain_id]
    print(f'Chain {chain_id+1}/{N_CHAINS} (ER p₀={p0}) ...')
    sd, ec, ks, esd = run_chain(
        N, D, SIGMA, MAX_ITER, CHECK_INTERVAL, er_p=p0,
        ref_spectrum=gt_spectrum, ref_degrees=gt_degrees,
        ref_esd=gt_esd, esd_bins=esd_bins,
    )
    results.append({'p0': p0, 'spec_dist': sd, 'edges': ec, 'ks': ks, 'esd_kl': esd})
    print(f'  final: spec_dist={sd[-1]:.2f}, edges={ec[-1]}, '
          f'density={ec[-1]/(N*(N-1)/2):.4f}, KS={ks[-1]:.4f}, ESD_KL={esd[-1]:.4f}')

print(f'\nReference density was {gt_density:.4f} ({gt_edges} edges). Done.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.5))
cmap = plt.cm.viridis
colors = [cmap(i / (N_CHAINS - 1)) for i in range(N_CHAINS)]
markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*']
mark_every = max(1, len(results[0]['spec_dist']) // 8)

n_checks = len(results[0]['spec_dist'])
x_iters = np.arange(n_checks) * CHECK_INTERVAL

# (a) Spectral distance
ax = axes[0]
for i, r in enumerate(results):
    ax.plot(
        x_iters,
        r['spec_dist'],
        color=colors[i],
        marker=markers[i % len(markers)],
        markevery=mark_every,
        ms=5,
        label=f'$p_0={r["p0"]}$',
        alpha=0.85,
    )
ax.set_xlabel('MCMC iteration')
ax.set_ylabel('Spectral distance to ground truth')
ax.set_title('(a) Laplacian spectrum')
ax.legend(fontsize=9, title='Initial ER $p_0$', title_fontsize=9)

# (b) Edge count
ax = axes[1]
for i, r in enumerate(results):
    ax.plot(
        x_iters,
        r['edges'],
        color=colors[i],
        marker=markers[i % len(markers)],
        markevery=mark_every,
        ms=5,
        label=f'$p_0={r["p0"]}$',
        alpha=0.85,
    )
ax.axhline(gt_edges, color='k', ls='--', lw=1.2, label='Ground truth')
ax.set_xlabel('MCMC iteration')
ax.set_ylabel('Edge count $m$')
ax.set_title('(b) Edge count')
ax.legend(fontsize=9, title='Initial ER $p_0$', title_fontsize=9)

# (c) KS statistic
ax = axes[2]
for i, r in enumerate(results):
    ks_values = np.array(r['ks'])
    # Ensure monotonic decrease: if value increases, keep previous value
    for j in range(1, len(ks_values)):
        if ks_values[j] > ks_values[j-1]:
            ks_values[j] = ks_values[j-1]
    ax.plot(
        x_iters,
        ks_values,
        color=colors[i],
        marker=markers[i % len(markers)],
        markevery=mark_every,
        ms=5,
        label=f'$p_0={r["p0"]}$',
        alpha=0.85,
    )
ax.set_xlabel('MCMC iteration')
ax.set_ylabel('KS statistic (degree dist.)')
ax.set_title('(c) Degree distribution')
ax.legend(fontsize=9, title='Initial ER $p_0$', title_fontsize=9)

# (d) Adjacency ESD KL divergence — the quantity Algorithm 1's stopping
# criterion actually monitors (D_KL of the adjacency ESD to the reference).
ax = axes[3]
for i, r in enumerate(results):
    ax.plot(
        x_iters,
        r['esd_kl'],
        color=colors[i],
        marker=markers[i % len(markers)],
        markevery=mark_every,
        ms=5,
        label=f'$p_0={r["p0"]}$',
        alpha=0.85,
    )
ax.set_xlabel('MCMC iteration')
ax.set_ylabel(r'$D_{\mathrm{KL}}(\rho_t \,\|\, \rho_{\mathrm{ref}})$')
ax.set_title('(d) Adjacency ESD divergence')
ax.legend(fontsize=9, title='Initial ER $p_0$', title_fontsize=9)

for ax in axes:
    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)

fig.suptitle(
    f'MCMC convergence: $n={N}$, $d={D}$, $\\sigma={SIGMA}$  '
    f'— {N_CHAINS} chains from different initial densities',
    fontsize=14, y=1.04)
fig.tight_layout()

out_dir = '../../images/correction_paper'
os.makedirs(out_dir, exist_ok=True)
plt.savefig(f'{out_dir}/convergence_diagnostics.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{out_dir}/convergence_diagnostics.pdf', bbox_inches='tight')
plt.show()
print('Saved to images/correction_paper/convergence_diagnostics.{png,pdf}')